In [1]:
# Install necessary libraries
!pip install --quiet requests folium pandas branca

In [2]:
import requests
import folium
import pandas as pd
from folium.plugins import HeatMap

In [3]:
# Geospatial Data Visualization
# By AHABWE ARON

import requests
import folium
import pandas as pd
from folium.plugins import HeatMap

# Step 1: Fetch earthquake data from the USGS API
# use "all_day" feed for simplicity (last 24 hours)
url = "https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_day.geojson"
response = requests.get(url)
data = response.json()

# Step 2: Convert earthquake data into a pandas DataFrame
features = data["features"]
earthquakes = []

for f in features:
    props = f["properties"]
    coords = f["geometry"]["coordinates"]
    earthquakes.append({
        "place": props["place"],
        "magnitude": props["mag"],
        "time": pd.to_datetime(props["time"], unit="ms"),
        "depth": coords[2],
        "longitude": coords[0],
        "latitude": coords[1]
    })

df = pd.DataFrame(earthquakes)

# Step 3: Create a base map
m = folium.Map(location=[0, 0], zoom_start=2)

# Step 4: Add earthquake markers
for _, row in df.iterrows():
    # Color based on depth
    if row["depth"] < 50:
        color = "green"
    elif row["depth"] < 150:
        color = "orange"
    else:
        color = "red"

    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=row["magnitude"] * 2 if row["magnitude"] else 2,
        color=color,
        fill=True,
        fill_opacity=0.6,
        popup=f"<b>Location:</b> {row['place']}<br>"
              f"<b>Magnitude:</b> {row['magnitude']}<br>"
              f"<b>Depth:</b> {row['depth']} km<br>"
              f"<b>Time:</b> {row['time']}"
    ).add_to(m)

# Step 5: Add heatmap layer
heat_data = [[row["latitude"], row["longitude"]] for _, row in df.iterrows()]
HeatMap(heat_data).add_to(m)

# Step 6: Display summary stats
print("Summary of Earthquakes (Last 24 Hours):")
print(df.describe())

# Step 7: Show map
m

Summary of Earthquakes (Last 24 Hours):
        magnitude                           time       depth   longitude  \
count  240.000000                            240  240.000000  240.000000   
mean     1.686077  2026-04-10 00:27:45.995837696   23.058522 -117.614004   
min     -1.500000     2026-04-09 12:38:17.019000   -1.860000 -178.501000   
25%      0.867500  2026-04-09 18:12:54.558749952    3.847500 -153.303250   
50%      1.600000  2026-04-10 00:41:28.555500032    7.839950 -122.808582   
75%      2.200000  2026-04-10 06:16:40.692500224   20.050000 -110.715250   
max      5.200000     2026-04-10 12:28:34.210000  535.946000  157.356400   
std      1.237539                            NaN   55.422956   61.361947   

         latitude  
count  240.000000  
mean    38.351664  
min    -60.992300  
25%     31.616000  
50%     36.324083  
75%     55.626542  
max     65.817000  
std     18.832018  
